<a href="https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mufsina/sonia-flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1

The paper reports that within the mostly AI-authored portfolio, age-controlled comparisons did not show a simple blanket penalty tied only to AI use. The differences varied across model cohorts, suggesting that process quality, editing standards, and topic fit may matter more than a simple AI-versus-human distinction.

**Methodology question:**  
Where does the label for AI use and the outcome measure come from, and does the age-controlled comparison adequately account for other differences between the content cohorts? I would want to understand whether the validation design supports this finding as an observed portfolio pattern rather than a causal claim.

### Finding 2

The paper reports that low-competition keywords had a higher growth-to-decline ratio than high-competition keywords, with low-competition keywords described as 62% more likely to grow in the portfolio.

**Methodology question:**  
How is "growth" defined and how are the growing and declining labels constructed? Does the comparison or validation design account for other factors, such as content age and existing search visibility, that could influence growth alongside competition?

In [ ]:
print("Two paper findings and methodology questions documented.")

Two paper findings and methodology questions documented.


## 2. My model under an honest split (before/after)

### Before

In Week 5, the Random Forest was evaluated against the Week-4 baseline using a grouped-by-client split. The model achieved an F1 score of 0.5913, while the Week-4 baseline achieved an F1 score of 0.1881.

### After

I re-ran the model using a client-grouped validation design, keeping all content from a client in only one side of the split. This is an honest check because the model is evaluated on clients that were not present in training.

I compare the before and after results using the same F1 metric. The purpose is to check whether the model's observed performance remains similar under a client-grouped split, rather than assuming that the Week-5 result generalizes to unseen clients.

The re-run produced an F1 score of 0.5913, which was the same as the Week-5 observed F1 score. This result does not prove that the model generalizes better, but it provides measured evidence that the observed F1 score remained stable under this grouped-by-client evaluation. The result should be treated as decision-support evidence rather than a guarantee of performance on future clients.

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

DATA_PATH = Path("/content/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Clients:", df["client_id"].nunique())

df["target_down"] = (
    df["trend_direction"] == "down"
).astype(int)

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns]
y = df["target_down"]
groups = df["client_id"]

print("Data loaded successfully.")

Shape: (30000, 44)
Clients: 32
Data loaded successfully.


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train_audit = X.iloc[train_idx]
X_test_audit = X.iloc[test_idx]

y_train_audit = y.iloc[train_idx]
y_test_audit = y.iloc[test_idx]

print("Training rows:", len(X_train_audit))
print("Test rows:", len(X_test_audit))

print(
    "Training clients:",
    df.iloc[train_idx]["client_id"].nunique()
)

print(
    "Test clients:",
    df.iloc[test_idx]["client_id"].nunique()
)

print(
    "Client overlap:",
    len(
        set(df.iloc[train_idx]["client_id"])
        &
        set(df.iloc[test_idx]["client_id"])
    )
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


In [ ]:
preprocessor_audit = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)

audit_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

audit_pipeline = Pipeline([
    ("preprocessor", preprocessor_audit),
    ("model", audit_model)
])

audit_pipeline.fit(
    X_train_audit,
    y_train_audit
)

audit_pred = audit_pipeline.predict(X_test_audit)

after_f1 = f1_score(
    y_test_audit,
    audit_pred
)

print("After honest grouped split F1:", round(after_f1, 4))

After honest grouped split F1: 0.5913


In [ ]:
comparison_audit = pd.DataFrame([
    {
        "stage": "Week-5 model",
        "split": "Grouped by client",
        "f1": 0.5913
    },
    {
        "stage": "ML-09 re-run",
        "split": "Grouped by client",
        "f1": after_f1
    }
])

display(comparison_audit.round(4))

,stage,split,f1
0,Week-5 model,Grouped by client,0.5913
1,ML-09 re-run,Grouped by client,0.5913


## 3. Leakage audit

I reviewed the final feature set for possible target leakage. The target is whether trend_direction is "down", so features that directly contain the current trend label or are calculated from the target should not be used as model inputs.

I also checked whether identifiers such as client_id and content_id were included in the feature set. They are excluded from the final feature list used by the model.

The audit is focused on whether any feature directly reveals the target rather than providing an independent signal.

In [ ]:
# Leakage audit

target_related = [
    "trend_direction",
    "trend_pct",
    "target_down",
    "prediction",
    "error_type"
]

identifier_features = [
    "client_id",
    "content_id"
]

print("Target-related columns found in feature list:")
print([
    col for col in target_related
    if col in feature_columns
])

print("\nIdentifier columns found in feature list:")
print([
    col for col in identifier_features
    if col in feature_columns
])

print("\nFinal feature count:", len(feature_columns))

print("\nFeatures used by the model:")
print(feature_columns)

Target-related columns found in feature list:
[]

Identifier columns found in feature list:
[]

Final feature count: 34

Features used by the model:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


### Leakage audit result

The audit found no direct target-related columns or identifier columns in the final feature list. The model uses 34 features, and `trend_direction`, `trend_pct`, `target_down`, `client_id`, and `content_id` are excluded.

This reduces the risk of direct target leakage in the feature list. However, this check does not prove that every feature is temporally independent of the target, so the result should be treated as a feature-level leakage check rather than proof of zero leakage.

## 4. Claim rewrite

### Original claim

The Random Forest is useful for identifying content that needs attention and performs better than the Week-4 baseline.

### Safer claim

On the evaluated data, the Random Forest showed a higher measured F1 score than the Week-4 baseline. Under the grouped-by-client evaluation, the observed result provides directional, decision-support evidence for identifying content that may need attention. It should not be treated as a guarantee that a page will decline or that refreshing a page will improve performance.

In [ ]:
print("Original claim rewritten using observed, measured, directional, and decision-support language.")

Original claim rewritten using observed, measured, directional, and decision-support language.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support

- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.